In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(f"./../")

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RXGate
from src.mcrx_simplifier import MCRXCascadeSimplifier
from src.misc import (
    get_state, 
    compare_quantum_states, 
    detailed_state_analysis,
    analyze_circuit_comparison,
    multi_crx
)

In [2]:
def safe_count_gates(circuit, optimization_level=3):
    """Count gates with robust error handling."""
    try:
        transpiled_circuit = transpile(
            circuit, 
            basis_gates=["cx", "u3"], 
            optimization_level=optimization_level
        )
        op_counts = transpiled_circuit.count_ops()
        cx_count = op_counts.get("cx", 0)
        u3_count = op_counts.get("u3", 0)
        return cx_count, u3_count, True
        
    except Exception as e:
        print(f"  Transpilation failed: {str(e)[:50]}...")
        
        # Fallback estimation
        try:
            total_gates = len(circuit.data)
            estimated_cx = max(1, total_gates * 4)
            estimated_u3 = max(1, total_gates * 2)
            return estimated_cx, estimated_u3, False
        except Exception:
            return 8, 4, False


def run_example_with_detailed_optimization(description, circuit_creation_func):
    """Run optimization example with comprehensive analysis."""
    print(f"\n{'='*80}")
    print(f"EXAMPLE: {description}")
    print(f"{'='*80}")
    
    # Initialize default return values
    qc_optimized = None
    cx_count_opt, u3_count_opt, transpiled_opt = 0, 0, False
    method_opt = "Failed"
    optimization_info = {}
    equivalence = "? UNKNOWN"
    state_results = None
    
    try:
        # Create original circuit
        qc_original = circuit_creation_func()
        
        print(f"\n--- Original Circuit ---")
        print(qc_original.draw())
        print(f"Original gates: {len(qc_original.data)}")
        print(f"Original depth: {qc_original.depth()}")
        
        # Get gate counts for original circuit
        cx_count_orig, u3_count_orig, transpiled_orig = safe_count_gates(qc_original)
        method_orig = "Transpiled" if transpiled_orig else "Estimated"
        print(f"Gate counts ({method_orig}) - CX: {cx_count_orig}, U3: {u3_count_orig}")
        
        # Initialize the simplifier
        simplifier = MCRXCascadeSimplifier(tolerance=1e-10, verbose=True)

        # Analyze optimization potential
        print(f"\n--- Optimization Analysis ---")
        try:
            # Check if the analyze method exists
            if hasattr(simplifier, 'analyze_patterns'):
                analysis = simplifier.analyze_patterns(qc_original)
                if analysis.get('status') == 'success':
                    reduction = analysis.get('reduction', 0)
                    reduction_percentage = (reduction / len(qc_original.data)) * 100 if len(qc_original.data) > 0 else 0
                    optimization_potential = "HIGH" if reduction_percentage > 50 else "MEDIUM" if reduction_percentage > 20 else "LOW"
                    print(f"Optimization potential: {optimization_potential}")
                    print(f"Expected reduction: {reduction_percentage:.1f}%")
                    print(f"XOR pairs found: {len(analysis.get('xor_pairs', []))}")
                else:
                    print(f"Analysis failed: {analysis.get('error', 'Unknown error')}")
            else:
                print("Pattern analysis not available")
        except Exception as e:
            print(f"Analysis failed: {e}")
        
        # Initialize optimization variables with defaults
        qc_optimized = qc_original
        cx_count_opt, u3_count_opt, transpiled_opt = cx_count_orig, u3_count_orig, transpiled_orig
        method_opt = method_orig
        
        # Perform optimization
        print(f"\n--- Applying Optimization ---")
        try:
            # Handle different API formats
            result = simplifier.simplify(qc_original)
            
            # Check if new API (returns tuple) or old API (returns circuit)
            if isinstance(result, tuple) and len(result) == 2:
                qc_optimized, optimization_info = result
                print(f"✓ New API detected - optimization info available")
            else:
                qc_optimized = result
                optimization_info = {}
                print(f"✓ Old API detected - basic optimization only")
            
            if qc_optimized is not None:
                print(f"\n--- Optimized Circuit ---")
                print(qc_optimized.draw())
                print(f"Optimized gates: {len(qc_optimized.data)}")
                print(f"Optimized depth: {qc_optimized.depth()}")
                
                # Print additional optimization details if available
                if optimization_info:
                    print(f"CNOT tricks used: {optimization_info.get('uses_cnot_tricks', False)}")
                    print(f"Total multi_crx calls: {optimization_info.get('total_multi_crx_calls', 'N/A')}")
                    if optimization_info.get('final_gates'):
                        print(f"Final gate calls: {optimization_info['final_gates']}")
                
                # Get gate counts for optimized circuit
                cx_count_opt, u3_count_opt, transpiled_opt = safe_count_gates(qc_optimized)
                method_opt = "Transpiled" if transpiled_opt else "Estimated"
                print(f"Gate counts ({method_opt}) - CX: {cx_count_opt}, U3: {u3_count_opt}")
            else:
                print("Optimization returned None - using original circuit")
                qc_optimized = qc_original
            
        except Exception as e:
            print(f"Optimization failed: {e}")
            qc_optimized = qc_original
        
        # State equivalence verification
        print(f"\n--- State Equivalence Verification ---")
        try:
            state_original = get_state(qc_original)
            state_optimized = get_state(qc_optimized)
            
            # Use state analysis
            state_results = compare_quantum_states(
                state_original, 
                state_optimized, 
                tolerance=1e-10, 
                verbose=True
            )
            
            if state_results['amplitudes_match']:
                print("✓ States are EQUIVALENT")
                equivalence = "✓ EQUIVALENT"
            else:
                print("✗ States are NOT equivalent")
                print(f"  Max difference: {state_results.get('max_difference', 'N/A')}")
                equivalence = "✗ NOT EQUIVALENT"
                
        except Exception as e:
            print(f"State verification failed: {e}")
            equivalence = "? VERIFICATION FAILED"
            state_results = {'fidelity': 0.0}
        
        # Calculate performance metrics
        print(f"\n--- Performance Summary ---")
        cx_reduction = max(0, cx_count_orig - cx_count_opt)
        u3_reduction = max(0, u3_count_orig - u3_count_opt)
        total_reduction = cx_reduction + u3_reduction
        
        cx_pct = (cx_reduction / max(1, cx_count_orig)) * 100
        u3_pct = (u3_reduction / max(1, u3_count_orig)) * 100
        total_pct = (total_reduction / max(1, cx_count_orig + u3_count_orig)) * 100
        
        depth_orig = qc_original.depth()
        depth_opt = qc_optimized.depth()
        depth_reduction = max(0, depth_orig - depth_opt)
        depth_pct = (depth_reduction / max(1, depth_orig)) * 100
        
        analysis_method = f"{method_orig}/{method_opt}"
        
        print(f"Analysis method: {analysis_method}")
        print(f"CX gate reduction: {cx_reduction} ({cx_pct:.1f}%)")
        print(f"U3 gate reduction: {u3_reduction} ({u3_pct:.1f}%)")
        print(f"Total gate reduction: {total_reduction} ({total_pct:.1f}%)")
        print(f"Depth reduction: {depth_reduction} ({depth_pct:.1f}%)")
        print(f"Fidelity: {state_results.get('fidelity', 0.0):.10f}")
        
        # Print detailed optimization summary if available
        if optimization_info and hasattr(simplifier, 'print_optimization_summary'):
            print(f"\n--- Detailed Optimization Summary ---")
            try:
                simplifier.print_optimization_summary(optimization_info)
            except Exception as e:
                print(f"Could not print detailed summary: {e}")
        
        return {
            'equivalence': equivalence,
            'gate_reduction': total_pct,
            'depth_reduction': depth_pct,
            'cx_reduction': cx_pct,
            'u3_reduction': u3_pct,
            'analysis_method': analysis_method,
            'optimization_info': optimization_info,
            'transpiled_counts': {
                'original': {'cx': cx_count_orig, 'u3': u3_count_orig},
                'optimized': {'cx': cx_count_opt, 'u3': u3_count_opt}
            }
        }
        
    except Exception as e:
        print(f"Critical error: {e}")
        return {
            'equivalence': "✗ FAILED",
            'gate_reduction': 0,
            'depth_reduction': 0,
            'cx_reduction': 0,
            'u3_reduction': 0,
            'analysis_method': "Failed",
            'optimization_info': {},
            'transpiled_counts': None
        }


In [3]:
# Example 1: Fully Disjoint Control Patterns
def create_disjoint_example():
    theta = np.pi / 4
    qc = QuantumCircuit(3)
    rx1 = multi_crx(theta, '10')
    rx2 = multi_crx(theta, '01')
    qc.append(rx1, [0, 1, 2])
    qc.append(rx2, [0, 1, 2])
    return qc

print("🧪 RUNNING ENHANCED EXAMPLES WITH TRANSPILED ANALYSIS")
result1 = run_example_with_detailed_optimization("Fully Disjoint Control Patterns ['10', '01']", create_disjoint_example)

🧪 RUNNING ENHANCED EXAMPLES WITH TRANSPILED ANALYSIS

EXAMPLE: Fully Disjoint Control Patterns ['10', '01']

--- Original Circuit ---
                           
q_0: ─────■──────────o─────
          │          │     
q_1: ─────o──────────■─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Transpiled) - CX: 16, U3: 17

--- Optimization Analysis ---
Optimization potential: LOW
Expected reduction: 0.0%
XOR pairs found: 1

--- Applying Optimization ---
🔬 Starting FIXED pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('10', coeff=1, qubits=[0, 1])
  ControlPattern('01', coeff=1, qubits=[0, 1])
✓ Original Boolean: (x0 & ~x1) | (x1 & ~x0)
✓ Simplified Boolean: (x0 & ~x1) | (x1 & ~x0)
✓ XOR pairs detected: 1
  XOR: 10 ⊕ 01 at positions [0, 1]
✓ Optimization: CX_trick
✓ Gate reduction: -1
✓ New API detected - optimization

In [4]:
# Example 2: Partially Overlapping Control Patterns  
def create_overlapping_example():
    theta = np.pi / 4
    qc = QuantumCircuit(3)
    rx1 = multi_crx(theta, '11')
    rx2 = multi_crx(theta, '10')
    qc.append(rx1, [0, 1, 2])
    qc.append(rx2, [0, 1, 2])
    return qc

result2 = run_example_with_detailed_optimization("Partially Overlapping Control Patterns ['11', '10']", create_overlapping_example)



EXAMPLE: Partially Overlapping Control Patterns ['11', '10']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────o─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Transpiled) - CX: 16, U3: 15

--- Optimization Analysis ---
Optimization potential: LOW
Expected reduction: 0.0%
XOR pairs found: 0

--- Applying Optimization ---
🔬 Starting FIXED pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('11', coeff=1, qubits=[0, 1])
  ControlPattern('10', coeff=1, qubits=[0, 1])
✓ Original Boolean: (x0 & x1) | (x0 & ~x1)
✓ Simplified Boolean: x0
✓ XOR pairs detected: 0
✓ Optimization: single_control
✓ Gate reduction: 1
✓ New API detected - optimization info available

--- Optimized Circuit ---
                
q_0: ─────■─────
          │     
q_1:

In [5]:
# Example 3: Complex Overlapping (3-qubit controls)
def create_complex_example():
    theta = 2 * np.pi / 4  # π/2
    qc = QuantumCircuit(4)
    rx1 = multi_crx(theta, '110')
    rx2 = multi_crx(theta, '101')
    qc.append(rx1, [0, 1, 2, 3])
    qc.append(rx2, [0, 1, 2, 3])
    return qc

result3 = run_example_with_detailed_optimization("Complex Overlapping Control Patterns ['110', '101']", create_complex_example)



EXAMPLE: Complex Overlapping Control Patterns ['110', '101']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────o─────
          │          │     
q_2: ─────o──────────■─────
     ┌────┴────┐┌────┴────┐
q_3: ┤ Rx(π/2) ├┤ Rx(π/2) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Transpiled) - CX: 40, U3: 33

--- Optimization Analysis ---
Optimization potential: LOW
Expected reduction: 0.0%
XOR pairs found: 1

--- Applying Optimization ---
🔬 Starting FIXED pattern reading MCRX simplification...
✓ Circuit validated: target=3, angle=1.5708, controls=3
✓ Extracted 2 patterns:
  ControlPattern('110', coeff=1, qubits=[0, 1, 2])
  ControlPattern('101', coeff=1, qubits=[0, 1, 2])
✓ Original Boolean: (x0 & x1 & ~x2) | (x0 & x2 & ~x1)
✓ Simplified Boolean: x0 & (x1 | x2) & (~x1 | ~x2)
✓ XOR pairs detected: 1
  XOR: 110 ⊕ 101 at positions [1, 2]
✓ Optimization: CX_trick
✓ Gate reduction: -

In [6]:
# Example 4: Subset Control Patterns
def create_subset_example():
    theta = np.pi / 4
    qc = QuantumCircuit(3)
    rx1 = multi_crx(theta, '11')
    qc.append(rx1, [0, 1, 2])
    # Fixed: Use MCRX instead of CRX to maintain same target requirement
    rx2 = multi_crx(theta, '1')  # This is effectively a '1X' pattern where X can be 0 or 1
    qc.append(rx2, [0, 2])  # Only qubit 0 as control, same target qubit 2
    return qc

result4 = run_example_with_detailed_optimization("Subset Control Patterns ['11'] + MCRX('1', 0->2)", create_subset_example)



EXAMPLE: Subset Control Patterns ['11'] + MCRX('1', 0->2)

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────┼─────
     ┌────┴────┐┌────┴────┐
q_2: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Transpiled) - CX: 8, U3: 10

--- Optimization Analysis ---
Optimization potential: LOW
Expected reduction: 0.0%
XOR pairs found: 0

--- Applying Optimization ---
🔬 Starting FIXED pattern reading MCRX simplification...
✓ Circuit validated: target=2, angle=0.7854, controls=2
✓ Extracted 2 patterns:
  ControlPattern('11', coeff=1, qubits=[0, 1])
  ControlPattern('1', coeff=1, qubits=[0])
✓ Original Boolean: x0 | (x0 & x1)
✓ Simplified Boolean: x0
✓ XOR pairs detected: 0
✓ Optimization: single_control
✓ Gate reduction: 1
✓ New API detected - optimization info available

--- Optimized Circuit ---
                
q_0: ─────■─────
          │     
q_1: ─────┼─────
   

In [7]:
# Example 5: Identical Control Patterns
def create_identical_example():
    theta = np.pi / 4
    qc = QuantumCircuit(4)
    rx1 = multi_crx(theta, '110')
    rx2 = multi_crx(theta, '110')
    qc.append(rx1, [0, 1, 2, 3])
    qc.append(rx2, [0, 1, 2, 3])
    return qc

result5 = run_example_with_detailed_optimization("Identical Control Patterns ['110', '110']", create_identical_example)


EXAMPLE: Identical Control Patterns ['110', '110']

--- Original Circuit ---
                           
q_0: ─────■──────────■─────
          │          │     
q_1: ─────■──────────■─────
          │          │     
q_2: ─────o──────────o─────
     ┌────┴────┐┌────┴────┐
q_3: ┤ Rx(π/4) ├┤ Rx(π/4) ├
     └─────────┘└─────────┘
Original gates: 2
Original depth: 2
Gate counts (Transpiled) - CX: 40, U3: 31

--- Optimization Analysis ---
Optimization potential: LOW
Expected reduction: 0.0%
XOR pairs found: 0

--- Applying Optimization ---
🔬 Starting FIXED pattern reading MCRX simplification...
✓ Circuit validated: target=3, angle=0.7854, controls=3
✓ Extracted 1 patterns:
  ControlPattern('110', coeff=2, qubits=[0, 1, 2])
✓ Original Boolean: x0 & x1 & ~x2
✓ Simplified Boolean: x0 & x1 & ~x2
✓ XOR pairs detected: 0
✓ Optimization: identical_patterns
✓ Gate reduction: 1
✓ New API detected - optimization info available

--- Optimized Circuit ---
                
q_0: ─────■─────
          │ 